In [3]:
import pandas as pd

# Load the dataset
df = pd.read_excel('PS 3.xlsx')
df

,ambient_temp_c,rel_humidity_pct,wind_velocity_kmh,precip_intensity_pct,cloud_state,atm_pressure_hpa,uv_radiation_idx,annual_phase,visibility_range_km,terrain_category,env_condition_label (Target)
0,14,73,9.5,82,partly cloudy,1010.82,2,Winter,3.5,inland,Rainy
1,39,96,8.5,71,partly cloudy,1011.43,7,Spring,10.0,inland,Cloudy
2,30,64,7.0,16,clear,1018.72,5,Spring,5.5,mountain,Sunny
3,38,83,1.5,82,clear,1026.25,7,Spring,1.0,coastal,Sunny
4,27,74,17.0,66,overcast,990.67,1,Winter,2.5,mountain,Rainy
...,...,...,...,...,...,...,...,...,...,...,...
13195,10,74,14.5,71,overcast,1003.15,1,Summer,1.0,mountain,Rainy
13196,-1,76,3.5,23,cloudy,1067.23,1,Winter,6.0,coastal,Snowy
13197,30,77,5.5,28,overcast,1012.69,3,Autumn,9.0,coastal,Cloudy
13198,3,76,10.0,94,overcast,984.27,0,Winter,2.0,inland,Snowy


In [6]:
import numpy as np
from collections import Counter

class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _gini(self, y):
        proportions = np.bincount(y) / len(y)
        return 1.0 - np.sum(proportions**2)

    def _best_split(self, X, y, feat_indices):
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_indices:
            thresholds = np.unique(X[:, feat_idx])
            for threshold in thresholds:
                # Split indices
                left_idx = np.where(X[:, feat_idx] <= threshold)[0]
                right_idx = np.where(X[:, feat_idx] > threshold)[0]

                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue

                # Calculate Information Gain (Reduction in Gini)
                n = len(y)
                n_l, n_r = len(left_idx), len(right_idx)
                gini_parent = self._gini(y)
                gini_children = (n_l/n)*self._gini(y[left_idx]) + (n_r/n)*self._gini(y[right_idx])
                gain = gini_parent - gini_children

                if gain > best_gain:
                    best_gain, split_idx, split_thresh = gain, feat_idx, threshold
        
        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Stopping criteria
        if depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split:
            leaf_value = Counter(y).most_common(1)[0][0]
            return {"leaf": True, "value": leaf_value}

        feat_indices = np.random.choice(n_features, int(np.sqrt(n_features)), replace=False)
        idx, thresh = self._best_split(X, y, feat_indices)

        left_idx = np.where(X[:, idx] <= thresh)[0]
        right_idx = np.where(X[:, idx] > thresh)[0]
        
        left_node = self._build_tree(X[left_idx], y[left_idx], depth + 1)
        right_node = self._build_tree(X[right_idx], y[right_idx], depth + 1)
        return {"leaf": False, "index": idx, "threshold": thresh, "left": left_node, "right": right_node}

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_one(self, x, node):
        if node["leaf"]: return node["value"]
        if x[node["index"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        return self._predict_one(x, node["right"])

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

In [7]:
class RandomForestScratch:
    def __init__(self, n_trees=10, max_depth=10):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth)
            # Bootstrapping: Random sample with replacement
            indices = np.random.choice(X.shape[0], X.shape[0], replace=True)
            tree.fit(X[indices], y[indices])
            self.trees.append(tree)

    def predict(self, X):
        # Majority voting
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Transpose to get predictions per sample and take the mode
        return np.array([Counter(tree_preds[:, i]).most_common(1)[0][0] for i in range(X.shape[1])])

In [8]:
import matplotlib.pyplot as plt

# Assuming X_train, y_train, X_test, y_test are prepared from your Excel file
n_trees_list = [1, 5, 10, 20, 50]
accuracies = []

for n in n_trees_list:
    model = RandomForestScratch(n_trees=n)
    model.fit(X_train.values, y_train)
    preds = model.predict(X_test.values)
    acc = np.mean(preds == y_test)
    accuracies.append(acc)
    print(f"Trees: {n}, Accuracy: {acc:.4f}")

# Plotting the improvement curve
plt.figure(figsize=(10,6))
plt.plot(n_trees_list, accuracies, marker='o', linestyle='--', color='darkorange')
plt.title("Random Forest Accuracy Improvement (Ensemble Effect)")
plt.xlabel("Number of Trees")
plt.ylabel("Testing Accuracy")
plt.grid(True)
plt.show()

TypeError: '<=' not supported between instances of 'int' and 'NoneType'